# miRBioGenAtlas (v2.0.0) — Reproducible Multi-Modal Pipeline

**Resource:** Sample-level and aggregated copy-number alteration (CNA), somatic mutation, structural variant, and matched mRNA expression dataset for 15 microRNA (miRNA) biogenesis and regulatory genes across 32 TCGA PanCancer Atlas (2018) studies (33 cancer types; $N = 10,195$ tumours).

### Portability & Execution Modes
- All file paths are resolved dynamically relative to the repository root via `pathlib.Path` (no hard-coded user directories).
- **`EXECUTION_MODE = "FROZEN_SNAPSHOT"` (default):** Verifies SHA-256 checksums of the archived cBioPortal v7.2.3 (DB v3.1.0) raw API responses in `data/raw_snapshot/` and deterministically rebuilds every sample-level, variant-level, patient-burden, aggregated, and validation table offline in ~8 seconds.
- **`EXECUTION_MODE = "LIVE_CBIOPORTAL_API"`:** Queries `https://www.cbioportal.org/api` live (`scripts/fetch_cbioportal_snapshot.py`) to refresh `data/raw_snapshot/` before running the identical deterministic harmonisation pipeline (`scripts/build_mirbiogen_v2_dataset.py`).

In [1]:
import csv
import gzip
import hashlib
import json
import pathlib
import subprocess
import sys

# Resolve repository root portably whether launched from repo root or notebooks/
CWD = pathlib.Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = REPO_ROOT / "data"
SNAPSHOT_DIR = DATA_DIR / "raw_snapshot"

EXECUTION_MODE = "FROZEN_SNAPSHOT"  # Set to "LIVE_CBIOPORTAL_API" to re-pull from cBioPortal

print(f"Python version : {sys.version.split()[0]}")
print(f"Repository root: {REPO_ROOT.name}/")
print(f"Execution mode : {EXECUTION_MODE}")

Python version : 3.9.6
Repository root: miRBioGenAtlas/
Execution mode : FROZEN_SNAPSHOT


## Step 1: Verify Frozen Input Snapshot (or Refresh from Live API)

In [2]:
if EXECUTION_MODE == "LIVE_CBIOPORTAL_API":
    subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "fetch_cbioportal_snapshot.py")], check=True)

manifest_path = SNAPSHOT_DIR / "snapshot_manifest.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

verified = 0
total_bytes = 0
for item in manifest["files"]:
    fpath = REPO_ROOT / item["path"]
    digest = hashlib.sha256(fpath.read_bytes()).hexdigest()
    assert digest == item["sha256"], f"Checksum mismatch: {item['path']}"
    verified += 1
    total_bytes += item["size_bytes"]

print(f"cBioPortal release : {manifest['cbioportal_portal_version']} (DB {manifest['cbioportal_db_version']})")
print(f"Snapshot retrieved : {manifest['retrieved_utc']}")
print(f"SHA-256 verified   : {verified}/{len(manifest['files'])} archive files ({total_bytes / (1024*1024):.2f} MB compressed)")

cBioPortal release : v7.2.3 (DB 3.1.0)
Snapshot retrieved : 2026-11-18T04:52:53Z
SHA-256 verified   : 33/33 archive files (9.06 MB compressed)


## Step 2: Execute Deterministic Sample-Level & Aggregated Harmonisation Pipeline

In [3]:
res = subprocess.run(
    [sys.executable, str(REPO_ROOT / "scripts" / "build_mirbiogen_v2_dataset.py")],
    capture_output=True,
    text=True,
    check=True,
)
print(res.stdout)

1. Verifying SHA-256 checksums of frozen raw_snapshot ...
   Snapshot verified: 33 files | cBioPortal v7.2.3 (DB 3.1.0)
2. Processing 32 TCGA PanCancer Atlas studies into sample-level & aggregated tables ...
3. Writing sample-level, variant-level, and patient-burden tables ...

=== SUMMARY FOR MANUSCRIPT & TECHNICAL VALIDATION ===
Total intersection samples (N): 10195
Total sample x gene rows: 152925
Total structural variant / fusion calls across 15 genes: 72
Total mRNA-matched samples (AGO2): 9620
Total hypermutated samples across 32 studies: 1385
Total annotated non-synonymous variant calls across 15 genes: 6292


## Step 3: Internal Consistency & Schema Verification Across Exported Files

In [4]:
# 1. Verify sample-level table row count (10,195 samples x 15 genes = 152,925 rows)
with gzip.open(DATA_DIR / "tcga_pancan_miRNA_biogenesis_sample_level_alterations.csv.gz", "rt", encoding="utf-8") as f:
    sl_reader = csv.DictReader(f)
    sl_cols = sl_reader.fieldnames
    n_sl_rows = sum(1 for _ in sl_reader)

# 2. Verify master 480-row aggregated table union identity
with open(DATA_DIR / "tcga_pancan_miRNA_biogenesis_alteration_freqs_long_FINAL.csv", "r", encoding="utf-8") as f:
    long_rows = list(csv.DictReader(f))

union_ok = sum(
    int(r["any_alter_n"]) == int(r["mut_only_n"]) + int(r["cna_only_n"]) + int(r["mut_and_cna_n"])
    for r in long_rows
)

print(f"Sample-level table : {n_sl_rows:,} rows x {len(sl_cols)} columns (expected 152,925)")
print(f"Aggregated table   : {len(long_rows)} rows; union identity holds in {union_ok}/{len(long_rows)} rows")
assert n_sl_rows == 152925
assert union_ok == 480

print("\nExported CSV/GZIP files in data/:")
for p in sorted(DATA_DIR.iterdir()):
    if p.is_file():
        sz_kb = p.stat().st_size / 1024.0
        unit = f"{sz_kb/1024:.2f} MB" if sz_kb >= 1024 else f"{sz_kb:.1f} KB"
        print(f"  {p.name:<62} {unit:>10}")

Sample-level table : 152,925 rows x 24 columns (expected 152,925)
Aggregated table   : 480 rows; union identity holds in 480/480 rows
